# SC-10-Account-Abstraction - ERC-4337

[<< Precedent : DAO Governance](SC-9-DAO-Governance.ipynb) | [Retour au sommaire](../README.md) | [Suivant : LLM Assisted >>](SC-11-LLM-Assisted.ipynb)

---

## Objectifs d'apprentissage

1. Comprendre l'**account abstraction** (ERC-4337)
2. Creer des **UserOperations**
3. Implementer un **Smart Account** simple
4. Comprendre les **Paymasters**

### Prerequis

- [SC-1](../01-Solidity-Foundation/SC-3-Solidity-Basics.ipynb) a [SC-7](SC-9-DAO-Governance.ipynb) completes
- Comprendre le modele de compte Ethereum (EOA vs contrats)
- Notions sur ERC-4337 (optionnel, couvert dans le notebook)

### Duree estimee : 50 minutes

---

## 0. Connexion a la blockchain locale

Tous les contrats de ce notebook sont **compiles et deployes reellement** sur anvil.
Lancez `anvil` dans un terminal avant d'executer les cellules.


In [1]:
# Connection a anvil (blockchain locale Foundry)
# Prerequis: anvil en cours d'execution dans un terminal
import sys, os
try:
    from web3 import Web3
except ImportError:
    print("Installation requise : pip install web3")
    raise

try:
    import solcx
except ImportError:
    print("Installation requise : pip install py-solc-x")
    raise

# Import forge_helper pour compiler avec Foundry (supporte @openzeppelin, @account-abstraction)
sys.path.insert(0, os.path.abspath(".."))
from forge_helper import forge_compile, forge_compile_and_deploy

SOLC_VERSION = "0.8.28"
ANVIL_URL = "http://127.0.0.1:8545"

# Connexion
w3 = Web3(Web3.HTTPProvider(ANVIL_URL))
assert w3.is_connected(), f"Impossible de se connecter a {ANVIL_URL}. Lancez 'anvil' dans un terminal."

# Installer solc si necessaire
installed = [str(v) for v in solcx.get_installed_solc_versions()]
if SOLC_VERSION not in installed:
    solcx.install_solc(SOLC_VERSION)
solcx.set_solc_version(SOLC_VERSION)

deployer = w3.eth.accounts[0]
print(f"Connecte a anvil (chain {w3.eth.chain_id}), deployer: {deployer[:10]}...")


def compile_and_deploy(w3, source_code, deployer, *constructor_args):
    """Compiler et deployer un contrat Solidity (via solcx, sans imports externes)."""
    compiled = solcx.compile_source(
        source_code, output_values=["abi", "bin"], solc_version=SOLC_VERSION
    )
    contract_id, contract_interface = compiled.popitem()
    Contract = w3.eth.contract(
        abi=contract_interface["abi"], bytecode=contract_interface["bin"]
    )
    tx_hash = Contract.constructor(*constructor_args).transact({"from": deployer})
    receipt = w3.eth.wait_for_transaction_receipt(tx_hash)
    instance = w3.eth.contract(
        address=receipt.contractAddress, abi=contract_interface["abi"]
    )
    print(f"Deploye: {contract_id.split(':')[-1]} a {receipt.contractAddress}")
    return instance, receipt


Connecte a anvil (chain 31337), deployer: 0xf39Fd6e5...


---

## 1. Concepts ERC-4337

ERC-4337 permet de separer la logique des comptes des wallets externes.

| Composant | Description |
|-----------|-------------|
| **UserOperation** | Transaction utilisateur (comme transfer, swap) |
| **EntryPoint** | Contrat singleton qui traite les UserOps |
| **Smart Account** | Wallet intelligent (logique metier en contrat) |
| **Paymaster** | Sponsor de frais de gas |
| **Bundler** | Empaquete les UserOps en batch |

In [2]:
# UserOperation structure
USER_OP_STRUCT = '''
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.28;

struct UserOperation {
    address sender;
    uint256 nonce;
    bytes initCode;
    bytes callData;
    uint256 callGasLimit;
    uint256 verificationGasLimit;
    uint256 preVerificationGas;
    uint256 maxFeePerGas;
    uint256 maxPriorityFeePerGas;
    bytes paymasterAndData;
    bytes signature;
}
'''

print("Structure UserOperation (ERC-4337):")
print(USER_OP_STRUCT)
print("Chaque champ controle un aspect de la transaction :")
print("  sender          = adresse du Smart Account")
print("  nonce           = protection contre le replay")
print("  callData        = la transaction a executer")
print("  signature       = preuve d'autorisation")
print("  paymasterAndData = donnees du paymaster (si gas sponsorise)")

Structure UserOperation (ERC-4337):

// SPDX-License-Identifier: MIT
pragma solidity ^0.8.28;

struct UserOperation {
    address sender;
    uint256 nonce;
    bytes initCode;
    bytes callData;
    uint256 callGasLimit;
    uint256 verificationGasLimit;
    uint256 preVerificationGas;
    uint256 maxFeePerGas;
    uint256 maxPriorityFeePerGas;
    bytes paymasterAndData;
    bytes signature;
}

Chaque champ controle un aspect de la transaction :
  sender          = adresse du Smart Account
  nonce           = protection contre le replay
  callData        = la transaction a executer
  signature       = preuve d'autorisation
  paymasterAndData = donnees du paymaster (si gas sponsorise)


### Interprétation : Structure UserOperation ERC-4337

**Sortie obtenue** : Affichage de la structure `UserOperation` et explication des champs.

| Champ | Type | Description |
|-------|------|-------------|
| `sender` | address | Adresse du Smart Account (créateur de l'opération) |
| `nonce` | uint256 | Compteur d'opérations (anti-replay) |
| `initCode` | bytes | Bytecode de création du compte (si compte inexistant) |
| `callData` | bytes | Données de l'appel à exécuter |
| `callGasLimit` | uint256 | Gas maximum pour l'exécution de l'appel |
| `verificationGasLimit` | uint256 | Gas pour la validation de signature |
| `preVerificationGas` | uint256 | Gas pour les frais fixes (remboursement) |
| `maxFeePerGas` | uint256 | Prix max du gas (EIP-1559) |
| `maxPriorityFeePerGas` | uint256 | Pourboire au validateur (EIP-1559) |
| `paymasterAndData` | bytes | Adresse du paymaster + données de sponsoring |
| `signature` | bytes | Signature de l'utilisateur (preuve d'autorisation) |

**Points clés** :
1. ERC-4337 introduit une nouvelle abstraction de compte qui remplace les EOA (Externally Owned Accounts)
2. La `UserOperation` est une pseudo-transaction qui sera traitée par l'EntryPoint
3. Le `paymasterAndData` permet de sponsoriser les frais de gas (gasless transactions)

> **Note technique** : Contrairement à une transaction Ethereum classique, une UserOperation ne contient pas de `value` ni de `to` directs. Ces informations sont encodées dans le `callData`.

---

## 2. Smart Account Simple

Un smart account est un contrat qui peut executer des transactions.

In [3]:
# Simple Smart Account (ERC-4337)
# Ce contrat herite de BaseAccount (account-abstraction SDK).
# Il necessite un EntryPoint deploye pour fonctionner en production.
# Ici on compile avec Foundry pour verifier la validite du code.

SIMPLE_ACCOUNT = '''
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.28;

import "@account-abstraction/contracts/core/BaseAccount.sol";
import "@account-abstraction/contracts/core/Helpers.sol";
import "@openzeppelin/contracts/utils/cryptography/ECDSA.sol";
import "@openzeppelin/contracts/utils/cryptography/MessageHashUtils.sol";

contract SimpleAccount is BaseAccount {
    using ECDSA for bytes32;

    address public owner;
    IEntryPoint private immutable _entryPoint;

    event SimpleAccountInitialized(IEntryPoint indexed entryPoint, address indexed owner);

    constructor(address anOwner, IEntryPoint anEntryPoint) {
        owner = anOwner;
        _entryPoint = anEntryPoint;
    }

    function entryPoint() public view override returns (IEntryPoint) {
        return _entryPoint;
    }

    // Verification de signature
    function _validateSignature(
        PackedUserOperation calldata userOp,
        bytes32 userOpHash
    ) internal virtual override returns (uint256 validationData) {
        bytes32 hash = MessageHashUtils.toEthSignedMessageHash(userOpHash);
        if (owner != ECDSA.recover(hash, userOp.signature)) {
            return SIG_VALIDATION_FAILED;
        }
        return SIG_VALIDATION_SUCCESS;
    }

    // Autoriser owner en plus de l EntryPoint
    function _requireForExecute() internal view override {
        require(
            msg.sender == address(_entryPoint) || msg.sender == owner,
            "only entry point or owner"
        );
    }

    receive() external payable {}
}
'''

# Compilation avec Foundry (resout les imports @account-abstraction et @openzeppelin)
abi, bytecode = forge_compile(SIMPLE_ACCOUNT, "SimpleAccount")
print(f"Compilation reussie ! ABI: {len(abi)} fonctions/events")
print(f"Bytecode: {len(bytecode)} caracteres")
print()

# Afficher les fonctions publiques du contrat compile
print("Fonctions et events du SimpleAccount:")
for item in abi:
    if item.get('type') == 'function':
        inputs = ', '.join(f"{i['type']} {i['name']}" for i in item.get('inputs', []))
        print(f"  function {item['name']}({inputs})")
    elif item.get('type') == 'event':
        print(f"  event {item['name']}")

# Note: le deploiement necessite un EntryPoint reel.
print()
print("Note: deploiement impossible sans EntryPoint reel.")
print("Voir la version standalone ci-dessous pour la demo deployable.")


Compilation reussie ! ABI: 14 fonctions/events
Bytecode: 9982 caracteres

Fonctions et events du SimpleAccount:
  function entryPoint()
  function execute(address target, uint256 value, bytes data)
  function executeBatch(tuple[] calls)
  function getNonce()
  function owner()
  function validateUserOp(tuple userOp, bytes32 userOpHash, uint256 missingAccountFunds)
  event SimpleAccountInitialized

Note: deploiement impossible sans EntryPoint reel.
Voir la version standalone ci-dessous pour la demo deployable.


### Interprétation : Compilation du SimpleAccount avec Foundry

**Sortie obtenue** : Compilation réussie avec Foundry, affichage de l'ABI et des fonctions.

| Métrique | Valeur | Signification |
|----------|--------|---------------|
| ABI length | 14 fonctions/events | Interface complète du contrat |
| Bytecode length | 9982 caractères | ~5 ko de bytecode (contrat complexe) |
| Compilation status | SUCCESS | Le code est valide Solidity 0.8.28 |

**Analyse des fonctions principales** :

| Fonction | Description | Héritée de |
|----------|-------------|------------|
| `entryPoint()` | Retourne l'adresse de l'EntryPoint | BaseAccount |
| `execute()` | Exécute une transaction | BaseAccount |
| `executeBatch()` | Exécute plusieurs transactions en lot | BaseAccount |
| `validateUserOp()` | Valide la signature de l'utilisateur | BaseAccount |
| `owner()` | Retourne le propriétaire du compte | SimpleAccount |

**Points clés** :
1. `SimpleAccount` hérite de `BaseAccount` qui gère la logique ERC-4337
2. La fonction `_validateSignature()` implémente la vérification ECDSA
3. L'ABI contient les événements pour le suivi des opérations

> **Note technique** : La compilation avec Foundry (`forge_compile`) résout automatiquement les imports `@account-abstraction` et `@openzeppelin` depuis le répertoire `lib/` du projet Foundry.

### Version standalone deployable

Le contrat ci-dessus depend d'un EntryPoint deploye. Voici une version
autonome qui illustre les memes concepts (owner, execution, validation)
sans dependances externes.

In [4]:
# Version standalone du Smart Account (sans dependances ERC-4337)
# Deploiement reel sur anvil

STANDALONE_ACCOUNT = '''
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.28;

/// @title StandaloneSmartAccount
/// @notice Version simplifiee d un smart account ERC-4337 pour demonstration.
/// Illustre: owner, execution de transactions, batch, nonce.
contract StandaloneSmartAccount {
    address public owner;
    uint256 public nonce;

    event Executed(address indexed dest, uint256 value, bytes data);
    event OwnerChanged(address indexed oldOwner, address indexed newOwner);

    modifier onlyOwner() {
        require(msg.sender == owner, "not owner");
        _;
    }

    constructor(address _owner) {
        owner = _owner;
    }

    // Executer une transaction
    function execute(address dest, uint256 value, bytes calldata data) external onlyOwner {
        nonce++;
        (bool success, ) = dest.call{value: value}(data);
        require(success, "execution failed");
        emit Executed(dest, value, data);
    }

    // Executer un batch
    function executeBatch(
        address[] calldata dests,
        uint256[] calldata values,
        bytes[] calldata datas
    ) external onlyOwner {
        require(dests.length == values.length && dests.length == datas.length, "length mismatch");
        for (uint256 i = 0; i < dests.length; i++) {
            nonce++;
            (bool success, ) = dests[i].call{value: values[i]}(datas[i]);
            require(success, "batch call failed");
            emit Executed(dests[i], values[i], datas[i]);
        }
    }

    // Changer le owner (social recovery, etc.)
    function changeOwner(address newOwner) external onlyOwner {
        require(newOwner != address(0), "zero address");
        emit OwnerChanged(owner, newOwner);
        owner = newOwner;
    }

    receive() external payable {}
}
'''

# Deployer sur anvil
account, receipt = compile_and_deploy(w3, STANDALONE_ACCOUNT, deployer, deployer)

# Verifier le owner
print(f"Owner: {account.functions.owner().call()}")
print(f"Nonce: {account.functions.nonce().call()}")

# Envoyer de l ETH au smart account
tx = w3.eth.send_transaction({
    'from': deployer,
    'to': account.address,
    'value': w3.to_wei(1, 'ether')
})
w3.eth.wait_for_transaction_receipt(tx)
balance = w3.eth.get_balance(account.address)
print(f"Balance du smart account: {w3.from_wei(balance, 'ether')} ETH")


Deploye: StandaloneSmartAccount a 0x84eA74d481Ee0A5332c457a4d796187F6Ba67fEB
Owner: 0xf39Fd6e51aad88F6F4ce6aB8827279cffFb92266
Nonce: 0
Balance du smart account: 1 ETH


### Interprétation : Déploiement et test du StandaloneSmartAccount

**Sortie obtenue** : Contrat déployé avec succès sur anvil, état initial affiché.

| Métrique | Valeur | Signification |
|----------|--------|---------------|
| Adresse | `0x84eA74...67fEB` | Adresse du contrat sur anvil |
| Owner | `0xf39Fd6...92266` | Compte propriétaire (deployer) |
| Nonce | 0 | Compteur d'opérations initialisé à 0 |
| Balance | 1 ETH | Solde du smart account après envoi |

**Flux d'exécution** :
1. **Construction** : Le contrat est créé avec le `deployer` comme owner
2. **Funding** : Envoi de 1 ETH au smart account pour les futures transactions
3. **Vérification** : Lecture de l'état (`owner()`, `nonce()`, balance)

**Points clés** :
1. Le `nonce` est incrémenté à chaque appel de `execute()` ou `executeBatch()`
2. Le modificateur `onlyOwner` restreint l'exécution au propriétaire uniquement
3. Le smart account peut recevoir et stocker de l'ETH (fonction `receive()`)

> **Note technique** : Cette version standalone illustre les concepts ERC-4337 sans dépendances. En production, utilisez l'infrastructure complète avec EntryPoint et Bundler.

---

## 3. Paymaster

Un paymaster peut payer les frais de gas pour les utilisateurs.

In [5]:
# Verifying Paymaster (ERC-4337)
# Un paymaster peut sponsoriser les frais de gas.
# Ce contrat herite de BasePaymaster (account-abstraction SDK)
# et necessite un EntryPoint deploye.

PAYMASTER = '''
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.28;

import "@account-abstraction/contracts/core/BasePaymaster.sol";
import "@account-abstraction/contracts/core/Helpers.sol";
import "@openzeppelin/contracts/utils/cryptography/ECDSA.sol";
import "@openzeppelin/contracts/utils/cryptography/MessageHashUtils.sol";

contract VerifyingPaymaster is BasePaymaster {
    using ECDSA for bytes32;

    address public verifyingSigner;

    constructor(IEntryPoint _entryPoint, address _verifyingSigner)
        BasePaymaster(_entryPoint, msg.sender)
    {
        verifyingSigner = _verifyingSigner;
    }

    // Validation du paymaster
    function _validatePaymasterUserOp(
        PackedUserOperation calldata userOp,
        bytes32 userOpHash,
        uint256 maxCost
    ) internal override returns (bytes memory context, uint256 validationData) {
        (uint48 validUntil, uint48 validAfter, bytes calldata signature) =
            parsePaymasterAndData(userOp.paymasterAndData);

        bytes32 hash = keccak256(abi.encode(userOpHash, validUntil, validAfter));
        bytes32 ethSignedHash = MessageHashUtils.toEthSignedMessageHash(hash);

        if (verifyingSigner != ECDSA.recover(ethSignedHash, signature)) {
            return ("", _packValidationData(true, validUntil, validAfter));
        }

        return ("", _packValidationData(false, validUntil, validAfter));
    }

    function parsePaymasterAndData(bytes calldata paymasterAndData)
        internal pure
        returns (uint48 validUntil, uint48 validAfter, bytes calldata signature)
    {
        require(paymasterAndData.length >= 52, "invalid paymasterAndData");
        validUntil = uint48(bytes6(paymasterAndData[20:26]));
        validAfter = uint48(bytes6(paymasterAndData[26:32]));
        signature = paymasterAndData[32:];
    }
}
'''

# Compilation avec Foundry
abi_pm, bytecode_pm = forge_compile(PAYMASTER, "VerifyingPaymaster")
print(f"Compilation reussie ! ABI: {len(abi_pm)} fonctions/events")
print(f"Bytecode: {len(bytecode_pm)} caracteres")
print()

# Afficher les fonctions publiques
print("Fonctions et events du VerifyingPaymaster:")
for item in abi_pm:
    if item.get('type') == 'function':
        inputs = ', '.join(f"{i['type']} {i['name']}" for i in item.get('inputs', []))
        print(f"  function {item['name']}({inputs})")
    elif item.get('type') == 'event':
        print(f"  event {item['name']}")

print()
print("Note: deploiement impossible sans EntryPoint reel (interface ERC-165 requise).")
print("En production, le Paymaster est deploye apres l'EntryPoint.")


Compilation reussie ! ABI: 26 fonctions/events
Bytecode: 15108 caracteres

Fonctions et events du VerifyingPaymaster:
  function acceptOwnership()
  function addStake(uint32 unstakeDelaySec)
  function deposit()
  function entryPoint()
  function getDeposit()
  function owner()
  function pendingOwner()
  function postOp(uint8 mode, bytes context, uint256 actualGasCost, uint256 actualUserOpFeePerGas)
  function renounceOwnership()
  function transferOwnership(address newOwner)
  function unlockStake()
  function validatePaymasterUserOp(tuple userOp, bytes32 userOpHash, uint256 maxCost)
  function verifyingSigner()
  function withdrawStake(address withdrawAddress)
  function withdrawTo(address withdrawAddress, uint256 amount)
  event OwnershipTransferStarted
  event OwnershipTransferred

Note: deploiement impossible sans EntryPoint reel (interface ERC-165 requise).
En production, le Paymaster est deploye apres l'EntryPoint.


### Interprétation : Compilation du VerifyingPaymaster

**Sortie obtenue** : Compilation réussie, affichage de l'ABI complète du Paymaster.

| Métrique | Valeur | Signification |
|----------|--------|---------------|
| ABI length | 26 fonctions/events | Interface étendue (gestion de stake, dépôts, retraits) |
| Bytecode length | 15108 caractères | ~7.5 ko de bytecode |
| Compilation status | SUCCESS | Le code est valide |

**Analyse des fonctions clés du Paymaster** :

| Catégorie | Fonctions | Description |
|-----------|-----------|-------------|
| **Staking** | `addStake()`, `unlockStake()`, `withdrawStake()` | Gestion du stake de sécurité (anti-DOS) |
| **Dépôts** | `deposit()`, `getDeposit()`, `withdrawTo()` | Gestion des fonds de sponsoring |
| **Validation** | `validatePaymasterUserOp()` | Validation de la signature du paymaster |
| **Ownership** | `owner()`, `transferOwnership()`, `acceptOwnership()` | Gestion de la propriété du contrat |

**Points clés** :
1. Le `VerifyingPaymaster` hérite de `BasePaymaster` qui implémente la logique ERC-4337
2. La fonction `_validatePaymasterUserOp()` vérifie qu'un signateur autorisé a approuvé l'opération
3. Le stake est requis pour éviter les attaques DOS (le paymaster perd son stake s'il fraude)

> **Note technique** : En production, le paymaster doit être staké (généralement ~1 ETH) et déposé avec des fonds pour sponsoriser les gas des utilisateurs.

### Version standalone deployable

Version simplifiee d'un paymaster qui illustre le mecanisme de sponsoring
de gas sans dependre de l'infrastructure EntryPoint.

In [6]:
# Version standalone du Paymaster (sans dependances ERC-4337)

STANDALONE_PAYMASTER = '''
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.28;

/// @title StandalonePaymaster
/// @notice Demonstre le concept de sponsoring de gas.
/// Le signer autorise des operations en signant off-chain.
contract StandalonePaymaster {
    address public owner;
    address public verifyingSigner;
    mapping(address => uint256) public deposits;

    event Deposited(address indexed account, uint256 amount);
    event Withdrawn(address indexed account, uint256 amount);
    event GasSponsored(address indexed account, uint256 amount);

    constructor(address _signer) {
        owner = msg.sender;
        verifyingSigner = _signer;
    }

    // Deposer des fonds pour sponsoriser le gas
    function deposit() external payable {
        deposits[msg.sender] += msg.value;
        emit Deposited(msg.sender, msg.value);
    }

    // Retirer des fonds
    function withdraw(uint256 amount) external {
        require(deposits[msg.sender] >= amount, "insufficient deposit");
        deposits[msg.sender] -= amount;
        payable(msg.sender).transfer(amount);
        emit Withdrawn(msg.sender, amount);
    }

    // Simuler le sponsoring de gas
    function sponsorGas(address account, uint256 gasAmount) external {
        require(msg.sender == owner, "only owner");
        require(deposits[owner] >= gasAmount, "insufficient funds");
        deposits[owner] -= gasAmount;
        emit GasSponsored(account, gasAmount);
    }

    function getDeposit(address account) external view returns (uint256) {
        return deposits[account];
    }
}
'''

# Deployer
paymaster, receipt = compile_and_deploy(w3, STANDALONE_PAYMASTER, deployer, deployer)

# Deposer des fonds
tx = paymaster.functions.deposit().transact({'from': deployer, 'value': w3.to_wei(5, 'ether')})
w3.eth.wait_for_transaction_receipt(tx)
deposit = paymaster.functions.getDeposit(deployer).call()
print(f"Deposit du paymaster: {w3.from_wei(deposit, 'ether')} ETH")

# Sponsoriser du gas pour un compte
user = w3.eth.accounts[1]
tx = paymaster.functions.sponsorGas(user, w3.to_wei(0.1, 'ether')).transact({'from': deployer})
w3.eth.wait_for_transaction_receipt(tx)
print(f"Gas sponsorise pour {user[:10]}... : 0.1 ETH")
deposit_after = paymaster.functions.getDeposit(deployer).call()
print(f"Deposit restant: {w3.from_wei(deposit_after, 'ether')} ETH")


Deploye: StandalonePaymaster a 0xa82fF9aFd8f496c3d6ac40E2a0F282E47488CFc9
Deposit du paymaster: 5 ETH
Gas sponsorise pour 0x70997970... : 0.1 ETH
Deposit restant: 4.9 ETH


### Interprétation : Déploiement et test du StandalonePaymaster

**Sortie obtenue** : Paymaster déployé, dépôt de 5 ETH, sponsoring de 0.1 ETH pour un utilisateur.

| Métrique | Valeur | Signification |
|----------|--------|---------------|
| Adresse | `0xa82fF9...CFc9` | Adresse du paymaster sur anvil |
| Deposit initial | 5 ETH | Fonds déposés pour le sponsoring |
| Gas sponsorisé | 0.1 ETH | Montant alloué à l'utilisateur |
| Deposit restant | 4.9 ETH | Solde après sponsoring |

**Flux de sponsoring de gas** :
1. **Dépôt** : Le propriétaire dépose des fonds (`deposit()`) pour sponsoriser les gas
2. **Sponsoring** : Le propriétaire appelle `sponsorGas(user, amount)` pour allouer des fonds
3. **Utilisation** : L'utilisateur peut utiliser ces fonds pour payer ses gas fees
4. **Retrait** : Les fonds non utilisés peuvent être retirés (`withdraw()`)

**Points clés** :
1. Le paymaster agit comme un intermédiaire qui paie les gas au nom des utilisateurs
2. Cette architecture permet des "gasless transactions" (l'utilisateur ne paie pas directement)
3. En production, le paymaster vérifie la signature off-chain avant de sponsoriser

> **Note technique** : Dans un vrai déploiement ERC-4337, le paymaster rembourse le smart account après l'exécution de la UserOperation. Le montant remboursé est basé sur le gas réellement utilisé.

---

## 4. Flux de Transaction ERC-4337

In [7]:
# Flux ERC-4337
print("""
FLUX DE TRANSACTION ERC-4337

1. CREATION
   User -> Crée UserOperation -> Signe avec clé privée

2. SOUMISSION
   UserOp -> Bundler (via RPC ou API)

3. VALIDATION
   Bundler -> simulateValidation() sur EntryPoint
   EntryPoint -> appelle validateUserOp() sur Smart Account
   Smart Account -> vérifie signature

4. EXECUTION
   Bundler -> agrège plusieurs UserOps
   Bundler -> handleOps() sur EntryPoint
   EntryPoint -> execute les opérations validées

AVANTAGES:
- Wallets sans seed phrase (social recovery)
- Gas sponsoring (paymasters)
- Batch transactions
- Transactions sans ETH (pay with tokens)
- Logic programmable (2FA, spending limits)

ADDRESSES EntryPoint:
- Mainnet: 0x5FF137D4b0FDCD49DcA30c7CF57E578a026d2789
- Sepolia: 0x5FF137D4b0FDCD49DcA30c7CF57E578a026d2789
- Goerli: 0x5FF137D4b0FDCD49DcA30c7CF57E578a026d2789
""")


FLUX DE TRANSACTION ERC-4337

1. CREATION
   User -> Crée UserOperation -> Signe avec clé privée

2. SOUMISSION
   UserOp -> Bundler (via RPC ou API)

3. VALIDATION
   Bundler -> simulateValidation() sur EntryPoint
   EntryPoint -> appelle validateUserOp() sur Smart Account
   Smart Account -> vérifie signature

4. EXECUTION
   Bundler -> agrège plusieurs UserOps
   Bundler -> handleOps() sur EntryPoint
   EntryPoint -> execute les opérations validées

AVANTAGES:
- Wallets sans seed phrase (social recovery)
- Gas sponsoring (paymasters)
- Batch transactions
- Transactions sans ETH (pay with tokens)
- Logic programmable (2FA, spending limits)

ADDRESSES EntryPoint:
- Mainnet: 0x5FF137D4b0FDCD49DcA30c7CF57E578a026d2789
- Sepolia: 0x5FF137D4b0FDCD49DcA30c7CF57E578a026d2789
- Goerli: 0x5FF137D4b0FDCD49DcA30c7CF57E578a026d2789



---

## 5. Exercices

In [8]:
# Exercice: Social Recovery Account
# Version standalone (sans dependances ERC-4337) - a completer
# TODO etudiant : implementer un smart account avec recuperation sociale via des gardiens
# Indice : inspirez-vous du StandaloneSmartAccount (cell-4b) pour la structure de base
# Etape 1 : definir les variables d'etat (owner, guardians, threshold, recovery state)
# Etape 2 : implementer le constructor pour initialiser owner et guardians
# Etape 3 : implementer execute() pour les transactions du owner
# Etape 4 : implementer startRecovery() et approveRecovery() pour le vote des gardiens
# Etape 5 : implementer cancelRecovery() pour que le owner puisse annuler

EXERCISE_RECOVERY = '''
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.28;

/// @title SocialRecoveryAccount
/// @notice Smart account avec recuperation sociale via des gardiens.
/// Si le owner perd sa cle, les gardiens peuvent voter pour changer le owner.
contract SocialRecoveryAccount {
    address public owner;
    address[] public guardians;
    mapping(address => bool) public isGuardian;
    uint256 public guardianCount;
    uint256 public threshold;
    uint256 public nonce;

    // Recovery state
    address public pendingNewOwner;
    uint256 public recoveryApprovals;
    mapping(address => bool) public hasApprovedRecovery;

    event GuardianAdded(address indexed guardian);
    event RecoveryStarted(address indexed newOwner);
    event RecoveryApproved(address indexed guardian);
    event RecoveryCompleted(address indexed newOwner);
    event Executed(address indexed dest, uint256 value);

    modifier onlyOwner() {
        // TODO: etudiant - verifier que msg.sender est le owner
        _;
    }

    modifier onlyGuardian() {
        // TODO: etudiant - verifier que msg.sender est un gardien
        _;
    }

    constructor(
        address _owner,
        address[] memory _guardians,
        uint256 _threshold
    ) {
        // TODO: etudiant - initialiser owner, threshold et la liste des gardiens
        // Indice : verifier que threshold <= guardians.length et threshold > 0
        // Indice : verifier qu'il n'y a pas de doublons parmi les gardiens
    }

    // Executer une transaction (owner seulement)
    function execute(address dest, uint256 value, bytes calldata data) external onlyOwner {
        // TODO: etudiant - incrementer nonce, executer l'appel et emettre Executed
    }

    // Demarrer une recuperation (gardien seulement)
    function startRecovery(address newOwner) external onlyGuardian {
        // TODO: etudiant - verifier qu'aucune recuperation n'est en cours
        // TODO: etudiant - enregistrer pendingNewOwner, premier vote et emettre RecoveryStarted
    }

    // Approuver une recuperation (gardien seulement)
    function approveRecovery() external onlyGuardian {
        // TODO: etudiant - verifier qu'une recuperation est en cours
        // TODO: etudiant - enregistrer le vote du gardien
        // TODO: etudiant - si approvals >= threshold, transferer ownership et reset
        // Indice : ne pas oublier de reset pendingNewOwner et hasApprovedRecovery
    }

    // Annuler la recuperation (owner seulement)
    function cancelRecovery() external onlyOwner {
        // TODO: etudiant - verifier qu'une recuperation est en cours
        // TODO: etudiant - reset tout l'etat de recuperation
    }

    receive() external payable {}
}
'''

print("Exercice Social Recovery Account - a completer")
print("Implementez les fonctions marquees TODO dans EXERCISE_RECOVERY")


### Indice : Exercice Social Recovery Account

**Objectif** : Implémenter un smart account avec récupération sociale via des gardiens.

**Structure de données** :
- `guardians` : tableau d'adresses des gardiens de confiance
- `threshold` : nombre minimum de gardiens requis pour valider la récupération
- `pendingNewOwner` : adresse du nouveau propriétaire en attente de validation
- `recoveryApprovals` : compteur de votes pour la récupération en cours
- `hasApprovedRecovery` : mapping des gardiens ayant voté

**Logique à implémenter** :

1. **Constructor** :
   - Initialiser `owner` avec le paramètre `_owner`
   - Vérifier que `threshold <= guardians.length` et `threshold > 0`
   - Ajouter chaque gardien au tableau et au mapping `isGuardian`
   - Vérifier qu'il n'y a pas de doublons

2. **execute()** :
   - Vérifier que `msg.sender == owner` (modificateur `onlyOwner`)
   - Incrémenter `nonce`
   - Appeler `dest.call{value: value}(data)`
   - Émettre l'événement `Executed`

3. **startRecovery()** (gardien seulement) :
   - Vérifier que `pendingNewOwner == address(0)` (pas de récupération en cours)
   - Enregistrer `pendingNewOwner = newOwner`
   - Enregistrer le premier vote du gardien qui lance la récupération
   - Émettre `RecoveryStarted`

4. **approveRecovery()** (gardien seulement) :
   - Vérifier qu'une récupération est en cours (`pendingNewOwner != address(0)`)
   - Vérifier que le gardien n'a pas encore voté (`!hasApprovedRecovery[msg.sender]`)
   - Enregistrer le vote et incrémenter `recoveryApprovals`
   - Si `recoveryApprovals >= threshold` : transférer le ownership et reset l'état

5. **cancelRecovery()** (owner seulement) :
   - Vérifier qu'une récupération est en cours
   - Reset `pendingNewOwner`, `recoveryApprovals` et tous les `hasApprovedRecovery`

> **Conseil** : Utilisez des boucles `for` pour itérer sur les gardiens dans le constructor et pour reset les approvals dans `cancelRecovery()`.

---

## 6. Resume

| Concept | Description |
|---------|-------------|
| UserOperation | Structure de transaction ERC-4337 |
| EntryPoint | Singleton qui traite les UserOps |
| Smart Account | Wallet programmable |
| Paymaster | Sponsor de gas fees |
| Bundler | Empaqueteur de transactions |
| Social Recovery | Recuperation via gardiens |

---

**Notebook suivant** : [SC-11-LLM-Assisted](SC-11-LLM-Assisted.ipynb)

---

[<< Precedent : DAO Governance](SC-9-DAO-Governance.ipynb) | [Retour au sommaire](../README.md) | [Suivant : LLM Assisted >>](SC-11-LLM-Assisted.ipynb)